<a href="https://colab.research.google.com/github/Camille2306/new_movie_recommender/blob/main/MOD21_DuPortableAuCluster_enonce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MOD 2.1 — TP « Du portable au cluster »

**Jeu de données : DVF géolocalisées** (*demandes de valeurs foncières*) — les
mutations immobilières (ventes, mais aussi échanges, adjudications…) des cinq
dernières années en France, sauf en Alsace, en Moselle et à Mayotte. Ce sont des
données de l'administration fiscale (DGFiP), géolocalisées par data.gouv.fr et
publiées sous Licence Ouverte 2.0. Le TP utilise le **millésime 2025-12**
(transactions jusqu'à fin 2025, fichiers mis à jour le 25 mai 2026), figé : tout le
monde obtient les mêmes chiffres, quelle que soit la date de la séance.

> **Données personnelles.** Chaque ligne décrit un bien réel, vendu par des
> personnes réelles. Ne cherchez jamais à ré-identifier vendeurs ou acquéreurs
> (article R112 A-3 du Livre des procédures fiscales), et ne les republiez pas sur
> le web : elles ne doivent pas être indexées par les moteurs de recherche.

Sur la **chaîne de valeur de la donnée** du module, ce TP couvre l'étape
**Traitement** : collecte → stockage → **traitement** → valorisation → impacts.

| | |
|---|---|
| **Durée** | 3 h, dont les **15 dernières minutes** pour un QCM |
| **Outils** | Python et pandas — sur Colab, ou avec Jupyter ou VS Code sur votre machine |
| **Évaluation** | **QCM de 15 min en fin de séance, ordinateur fermé, sans document ni calculatrice.** Rien à rendre : pas de compte-rendu |
| **IA générative** | **non autorisée** pendant le TP |

Le QCM vérifie que vous avez **compris** ce que vous avez fait, pas que vous savez
écrire du code de mémoire. Il porte sur ce notebook et sur le cours préparatoire.

> **Désactiver l'IA générative.** Le TP se fait sans IA générative : le code et
> les justifications doivent être les vôtres.
>
> - **Colab** : les fonctions d'IA sont déjà masquées dans ce notebook. Vérifiez-le :
>   *Modifier → Paramètres du notebook → Assistance IA* ; la case *Masquer les
>   fonctionnalités d'IA générative* doit être cochée (en anglais : *Edit →
>   Notebook settings → AI assistance → Hide generative AI features*).
> - **VS Code** : ouvrez les paramètres (Ctrl+, ou ⌘+,), cherchez
>   `chat.disableAIFeatures` et cochez la case.
> - **Jupyter (Anaconda)** : n'activez aucun assistant (Jupyter AI, Anaconda Assistant…).
>
> N'utilisez pas non plus ChatGPT, Gemini, Copilot ni aucun autre outil d'IA dans un autre onglet.

> **Conseil.** Exécutez les cellules dans l'ordre. Quand une question demande une
> justification, répondez **en français, dans la cellule de texte prévue**, même si
> personne ne la corrige : savoir expliquer, c'est la meilleure préparation au QCM.
> Gardez sous la main les diapos *pandas Quick Reference* et *Reading an Error* du
> cours préparatoire.

---
## Partie 0 — Mise en route (10 min)

On travaille d'abord sur un seul département, le **Rhône (69)**, pour aller vite.

> Sur Mac, avec le Python de python.org : si le téléchargement échoue avec
> `CERTIFICATE_VERIFY_FAILED`, lancez une fois *Install Certificates.command*
> (dossier *Applications → Python 3.x*).

In [1]:
import pandas as pd
import time, urllib.request, urllib.error, pathlib

pd.set_option("display.width", 120)
print("pandas", pd.__version__)

pandas 2.2.3


In [2]:
MILLESIME = "2025-12"      # figé : tout le monde a les mêmes données
BASE = f"https://files.data.gouv.fr/geo-dvf/{MILLESIME}/csv/2023"

def telecharge(url, nom):
    "Télécharge le fichier s'il n'est pas déjà présent localement."
    chemin = pathlib.Path(nom)
    if not chemin.exists():
        print("téléchargement de", nom, "...")
        partiel = chemin.with_suffix(".part")
        try:
            urllib.request.urlretrieve(url, partiel)
        except urllib.error.HTTPError:     # millésime retiré du serveur : prendre le plus récent
            print("millésime", MILLESIME, "indisponible : version la plus récente, chiffres un peu différents")
            urllib.request.urlretrieve(url.replace(MILLESIME, "latest"), partiel)
        partiel.rename(chemin)             # seulement si le téléchargement est allé au bout
    print(nom, ":", round(chemin.stat().st_size / 1e6, 1), "Mo")
    return chemin

fichier_69 = telecharge(f"{BASE}/departements/69.csv.gz", "dvf69.csv.gz")

téléchargement de dvf69.csv.gz ...
dvf69.csv.gz : 2.1 Mo


---
## Partie 1 — Découvrir et nettoyer les données (45 min)

Aucune donnée réelle n'est directement exploitable. Cette partie est celle qu'on
sous-estime toujours : les data scientists y passent près de la moitié de leur
temps (45 % selon l'enquête *State of Data Science* d'Anaconda, 2020).

### 1.1 — Charger et regarder

Chargez le fichier dans un DataFrame `df`, puis affichez sa taille, ses colonnes,
le nombre de ventes distinctes (colonne `id_mutation`) et les cinq premières lignes.

> Un avertissement `DtypeWarning` peut s'afficher : pandas hésite sur le type de
> certaines colonnes. Ce n'est pas une erreur ; on y revient en partie 3.

In [5]:
df = pd.read_csv("dvf69.csv.gz")          # à compléter : pd.read_csv(...)
print("dimensions :", df.shape)
print("colonnes :", list(df.columns))
print("ventes distinctes :", df["id_mutation"].nunique())   # indice : df["id_mutation"].nunique()
print(df.head(5))            # afficher les 5 premières lignes

dimensions : (84968, 40)
colonnes : ['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation', 'valeur_fonciere', 'adresse_numero', 'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune', 'nom_commune', 'code_departement', 'ancien_code_commune', 'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle', 'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez', 'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale', 'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude']
ventes distinctes : 31988
   id_mutation date_mutation  numero_disposition        nature_mutation  valeur_fonciere  adresse_numero  \
0  2023-894771    2023-01-03                   1            

/tmp/ipykernel_2355/3532784557.py:1: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("dvf69.csv.gz")          # à compléter : pd.read_csv(...)


**Question 1.1.** Combien de lignes, de colonnes, de ventes distinctes ?
Que représente donc **une ligne** ?

*Votre réponse :* 84968 lignes, 40 colonnes, 31988 ventes distinctes, une ligne représente une partie d'une vente

### 1.2 — Une vente, plusieurs lignes

Exécutez la cellule : elle affiche toutes les lignes de la première vente du fichier
(même `id_mutation`).

In [6]:
premiere = df[df["id_mutation"] == df["id_mutation"].iloc[0]]
premiere[["id_mutation", "date_mutation", "valeur_fonciere", "type_local",
          "surface_reelle_bati", "lot1_numero"]]

,id_mutation,date_mutation,valeur_fonciere,type_local,surface_reelle_bati,lot1_numero
0,2023-894771,2023-01-03,335000.0,Appartement,65.0,123
1,2023-894771,2023-01-03,335000.0,Appartement,70.0,116


**Question 1.2.** Combien de logements cette vente comprend-elle ?
`valeur_fonciere` est-il le prix de chaque logement, ou celui de toute la vente ?
Quel prix au m² obtiendrait-on pour chaque logement en divisant `valeur_fonciere`
par sa surface, et quel est le vrai prix au m² de la vente ?

*Votre réponse :*comprend 2 logements. Valeur foncière = valeur totale ; il faut diviser par la surface totale

### 1.3 — Ce qui manque, ce qui pollue

Regardez la colonne `type_local` et la proportion de valeurs manquantes des
colonnes qui nous intéressent.

In [7]:
print(df["type_local"].value_counts(dropna=False), "\n")
interet = ["valeur_fonciere", "surface_reelle_bati", "nom_commune", "nature_mutation"]
print(...)        # pourcentage de valeurs manquantes par colonne
print("mémoire occupée :", ...)   # indice : df.memory_usage(deep=True)

type_local
Dépendance                                  33297
Appartement                                 21720
NaN                                         17354
Maison                                       7043
Local industriel. commercial ou assimilé     5554
Name: count, dtype: int64 

Ellipsis
mémoire occupée : Ellipsis


**Question 1.3.** La mémoire occupée est très supérieure à la taille du fichier,
même décompressé. Pourquoi ? (Repensez au « piège » signalé en cours.)

*Votre réponse :* Car c'est un fichier texte ? Car il y a plein de cases vides ?

### 1.4 — Nettoyer

On ne garde que ce qui a un sens pour une analyse de prix au m² :

1. uniquement les `nature_mutation` de type `"Vente"` (ce qui écarte aussi les
   logements neufs vendus sur plan, `"Vente en l'état futur d'achèvement"`) ;
2. uniquement les `type_local` `"Appartement"` ou `"Maison"` ;
3. **une ligne par logement, et seulement les ventes d'un seul logement** :
   `valeur_fonciere` est le prix de **toute** la vente, répété sur chacune de ses
   lignes (code fourni ; une maison apparaît aussi une fois par nature de culture
   de son terrain, d'où le premier dédoublonnage) ;
4. les lignes où le prix **et** la surface sont renseignés ;
5. une surface d'au moins 9 m² et un prix supérieur à 1 000 € ;
6. un prix au m² compris entre 200 et 20 000 € (valeurs aberrantes).

In [38]:
from numpy import True_
CLES_LOGEMENT = ["id_mutation", "id_parcelle", "type_local", "surface_reelle_bati", "lot1_numero"]

def nettoie(d):
    """Applique les 6 règles de nettoyage et ajoute la colonne prix_m2."""
    # à compléter : règles 1 et 2
    d = d[(d["nature_mutation"].isin(["Vente"])) & (d["type_local"].isin(["Appartement", "Maison"]))].copy()
    # règle 3 (fournie) : une ligne par logement, puis seulement les ventes d'un seul logement
    d = d.drop_duplicates(CLES_LOGEMENT)
    d = d.drop_duplicates("id_mutation", keep=False)
    # à compléter : règles 4 et 5, puis .copy() et la colonne d["prix_m2"], puis règle 6 (terminez par .copy())
    d = d[(d["surface_reelle_bati"] >= 9) & (d["valeur_fonciere"] >= 1000)].copy()
    d.insert(loc=0, column='prix_m2', value=d["valeur_fonciere"] / d["surface_reelle_bati"])
    d = d[(d["prix_m2"] <= 20000) & (d["prix_m2"] >= 200)].copy()
    return d


ventes = nettoie(df)
print(f"{len(df)} lignes brutes -> {len(ventes)} lignes exploitables")
ventes.head()

84968 lignes brutes -> 20236 lignes exploitables


,prix_m2,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
3,2394.427419,2023-894773,2023-01-03,1,Vente,296909.0,153.0,NaN,CHE DU MONT,0080,...,Maison,124.0,5.0,S,sols,NaN,NaN,124.0,4.724042,45.670613
5,3448.275862,2023-894774,2023-01-04,1,Vente,200000.0,77.0,NaN,CRS CHARLEMAGNE,1490,...,Appartement,58.0,3.0,NaN,NaN,NaN,NaN,NaN,4.821643,45.743032
6,6022.352941,2023-894775,2023-01-04,1,Vente,511900.0,136.0,NaN,AV PIERRE DUMOND,0740,...,Maison,85.0,4.0,S,sols,NaN,NaN,258.0,4.713363,45.747649
11,4934.375000,2023-894777,2023-01-05,1,Vente,157900.0,7.0,B,RUE JEAN CLAUDE MARTIN,0480,...,Appartement,32.0,1.0,NaN,NaN,NaN,NaN,NaN,4.724741,45.745033
13,4058.333333,2023-894778,2023-01-04,1,Vente,243500.0,26.0,NaN,RUE DES AQUEDUCS,0415,...,Appartement,60.0,3.0,NaN,NaN,NaN,NaN,NaN,4.795455,45.759150


**Question 1.4.** Vous venez de jeter une grande partie des lignes.

- Que contiennent principalement les lignes écartées ? (regardez `type_local`)
- Sans la règle 3, près de 2 400 lignes du Rhône dépasseraient 20 000 €/m² ;
  avec elle, une vingtaine. Pourquoi ? Appuyez-vous sur la vente de la question 1.2.
- Le seuil « prix au m² < 20 000 € » est un **choix**. Quel biais introduit-il ?
  Que se passerait-il si vous l'enleviez ?

*Votre réponse :* La plupart contient du vide ; car la surface serait uniquement celle du logement alors que le prix serait le prix total de la vente; Cela induit le biais de ne pas comptabiliser les logements extrèmement cher, qui risquerait de tirer la moyenne vers le haut mais d'être moins représentative.

---
## Partie 2 — Split, apply, combine (50 min)

C'est le motif universel vu en cours : **grouper** les lignes, **calculer** par
groupe, **rassembler**. En pandas : `groupby` puis `agg`.

### 2.1 — Prix médian par commune

Calculez, par commune, le **nombre de ventes** et le **prix médian au m²**.
Ne gardez que les communes ayant **au moins 50 ventes** (sinon la médiane n'a
pas de sens), et triez par prix décroissant.

> Dans DVF, Lyon n'apparaît pas sous ce nom : la ville figure sous la forme de ses
> 9 arrondissements (`Lyon 1er Arrondissement`… `Lyon 9e Arrondissement`).

In [43]:
par_commune = (ventes.groupby("nom_commune")["prix_m2"]
                      .agg(n="size", median="median")
                      .query("n >= 50").sort_values(by="median", ascending=False))     # filtrer n >= 50, puis trier

print("communes retenues :", len(par_commune))
par_commune.head(10).round(0)

communes retenues : 67


,n,median
nom_commune,,
Saint-Cyr-au-Mont-d'Or,68,5884.0
Saint-Didier-au-Mont-d'Or,75,5840.0
Lyon 6e Arrondissement,581,5797.0
Lyon 2e Arrondissement,414,5662.0
Lyon 1er Arrondissement,388,5411.0
Lyon 4e Arrondissement,479,5333.0
Charbonnières-les-Bains,62,4929.0
Lyon 3e Arrondissement,1353,4891.0
Dardilly,69,4850.0


**Question 2.1.** Pourquoi utilise-t-on la **médiane** plutôt que la **moyenne**
pour un prix immobilier ? Retenez bien cette question : on y revient en partie 3.

*Votre réponse :* Pour ne pas retenir les valeurs abérantes, des logements très cher qui ne reflètent pas la réalité des prix de l'endroit. Les appartements les plus chers sont de nombreuses fois plus chers que les logements médiants.

### 2.2 — Les extrêmes

In [44]:
print("--- 5 communes les plus chères ---")
print(par_commune.head(5).round(0), "\n")
print("--- 5 communes les moins chères ---")
print(par_commune.tail(5).round(0))

--- 5 communes les plus chères ---
                             n  median
nom_commune                           
Saint-Cyr-au-Mont-d'Or      68  5884.0
Saint-Didier-au-Mont-d'Or   75  5840.0
Lyon 6e Arrondissement     581  5797.0
Lyon 2e Arrondissement     414  5662.0
Lyon 1er Arrondissement    388  5411.0 

--- 5 communes les moins chères ---
                              n  median
nom_commune                            
Saint-Symphorien-sur-Coise   53  2312.0
Saint-Fons                  142  2257.0
Givors                      160  2188.0
Tarare                      136  1725.0
Thizy-les-Bourgs             70  1433.0


### 2.3 — Appartement contre maison

Comparez le prix médian au m² des appartements et des maisons, et le nombre de
ventes de chaque type.

In [48]:
# à compléter : grouper par type_local et agréger
par_type = (ventes.groupby("type_local")["prix_m2"]
                      .agg(n="size", median="median")
                      .query("n >= 1").sort_values(by="median", ascending=False))

print("types retenues :", len(par_type))
par_type.head(10).round(0)

types retenues : 2


,n,median
type_local,,
Appartement,14871,3971.0
Maison,5365,3929.0


### 2.4 — Évolution dans l'année

Agrégez par mois pour voir l'évolution du nombre de ventes.

In [53]:
#ventes["mois"] = pd.to_datetime("date_mutation").dt.month
ventes["date_mutation"] =pd.to_datetime(ventes["date_mutation"])
ventes.insert(loc=0, column='mois', value=ventes["date_mutation"].dt.month)   # indice : pd.to_datetime(...).dt.month
par_mois = (ventes.groupby("mois")["prix_m2"]
                      .agg(n="size", median="median")
                      .query("n >= 1").sort_values(by="median", ascending=False))

par_mois.round(0)

,n,median
mois,,
8,1256,4140.0
1,1692,4055.0
3,1888,4039.0
2,1562,4036.0
6,2087,3990.0
5,1681,3970.0
7,2321,3969.0
9,1941,3965.0
4,1550,3946.0


> **Arrêtez-vous une seconde.** Chacune de ces cellules a fait la même chose :
> découper les lignes en groupes, calculer une valeur par groupe, rassembler.
> Dans le vocabulaire de MapReduce, découper, c'est le **map** (produire des
> couples clé–valeur) puis le **shuffle** (réunir les lignes de même clé) ;
> calculer et rassembler, c'est le **reduce**. Le shuffle ne calcule rien : il
> déplace les lignes.
>
> **Vous écrivez du MapReduce depuis le début de la partie 2**, sur une seule
> machine et sans le savoir. La partie 3 va rendre cela explicite.

---
## Partie 3 — Le mur, et comment le franchir (60 min)

Jusqu'ici, 85 000 lignes : tout tient en mémoire. Passons à la France entière.

> **Gérez votre temps** : la séance se termine par le QCM. Si vous êtes en retard,
> lisez au moins les questions **3.3** et **3.5** : le QCM en reprend les idées.

### 3.1 — Mesurer avant de subir

Avant de charger un gros fichier, on estime. Mesurez le temps de chargement et la
mémoire occupée pour un nombre croissant de lignes, puis **extrapolez**.

In [58]:
mesures = []
for n in [5_000, 20_000, 50_000, len(df)]:
    t0 = time.time()
    echantillon = pd.read_csv("dvf.csv.gz", nrows=n)        # à compléter : charger n lignes (nrows=n)
    duree = time.time() - t0           # à compléter : temps écoulé depuis t0
    mesures.append({"lignes": n,
                    "secondes": round(duree, 2),
                    "Mo_en_memoire": echantillon.memory_usage(deep=True).sum() / 1e6})   # à compléter : mémoire en Mo (memory_usage(deep=True))

mesures = pd.DataFrame(mesures)
mesures["Mo_par_million_de_lignes"] = (mesures.Mo_en_memoire / mesures.lignes * 1e6).round(0)
mesures

/tmp/ipykernel_2355/1332374563.py:4: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  echantillon = pd.read_csv("dvf.csv.gz", nrows=n)        # à compléter : charger n lignes (nrows=n)


,lignes,secondes,Mo_en_memoire,Mo_par_million_de_lignes
0,5000,0.02,4.394132,879.0
1,20000,0.05,17.594814,880.0
2,50000,0.18,45.214619,904.0
3,5000,0.03,4.394132,879.0


**Question 3.1.** À partir de la dernière colonne, estimez la mémoire nécessaire pour :

- la France entière sur un an (environ **3,8 millions** de lignes) ;
- la France entière sur les **cinq années publiées** (environ **20 millions** de lignes).

Comparez à la RAM de votre machine. Que se passe-t-il ?  
*Votre réponse :*  pour la france entière : 3.8 * 880 = 3500Mo   ;  sur 5 ans : 20*880 = 17600 Mo ; rame de machine env 16 GB = 2 Go


### 3.2 — Charger la France entière (~100 Mo compressés)

Le téléchargement prend une à deux minutes. Deux réflexes d'économie et de
robustesse, à retenir pour toute votre vie professionnelle :

1. **`usecols`** — ne lire que les colonnes utiles. Ici 10 colonnes sur 40.
2. **`dtype`** — imposer le type des colonnes qui sont des **identifiants**.

> **Le piège du département corse.** Sans `dtype`, pandas devine le type colonne
> par colonne, morceau par morceau. Or `code_departement` vaut `"01"`, `"69"`…
> mais aussi **`"2A"` et `"2B"`** pour la Corse. Selon les morceaux, pandas
> hésite entre nombre et texte : le `"01"` peut devenir `1` (zéro initial perdu),
> un même département peut apparaître sous deux clés (`29` et `"29"`), et
> certaines versions de pandas (3.0.0 à 3.0.5) échouent même à la lecture.
> **Un code n'est jamais un nombre**, même quand il en a l'air — c'est vrai des
> départements, des codes postaux, des numéros SIRET et des références produit.

In [54]:
fichier_fr = telecharge(f"{BASE}/full.csv.gz", "dvf_france.csv.gz")

COLS = ["id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere", "nom_commune",
        "code_departement", "id_parcelle", "type_local", "surface_reelle_bati", "lot1_numero"]
TYPES = {"id_mutation": "str", "nom_commune": "str", "code_departement": "str",
         "id_parcelle": "str", "nature_mutation": "str", "type_local": "str",
         "lot1_numero": "str"}

t0 = time.time()
france = pd.read_csv(fichier_fr, usecols=COLS, dtype=TYPES)
print(f"{len(france):,} lignes chargées en {time.time() - t0:.1f} s")
print("mémoire :", round(france.memory_usage(deep=True).sum() / 1e6), "Mo",
      "(avec seulement", len(COLS), "colonnes sur 40)\n")

non_numeriques = sorted(c for c in france["code_departement"].dropna().unique()
                        if not c.isdigit())
print("départements dont le code n'est pas un nombre :", non_numeriques)

téléchargement de dvf_france.csv.gz ...
dvf_france.csv.gz : 101.5 Mo
3,817,426 lignes chargées en 9.7 s
mémoire : 1759 Mo (avec seulement 10 colonnes sur 40)

départements dont le code n'est pas un nombre : ['2A', '2B']


**Question 3.2.** Relancez mentalement le calcul **sans** `usecols`, avec les
40 colonnes. Quel ordre de grandeur de mémoire obtiendriez-vous ? Est-ce tenable
sur les cinq années publiées ?

*Votre réponse :* on ferait fois 4 sur la mémoire (environ 7200Mo) et donc sur 5 ans ça fait 7200*5 = 36 Go

### 3.3 — Traiter par morceaux : map puis reduce

On calcule maintenant le **prix moyen au m² par département**, mais **sans jamais
charger tout le fichier**. Le principe vu en cours :

- **map** : sur chaque morceau, on calcule des agrégats *partiels* — ici le
  **nombre** de ventes et la **somme** des prix au m² par département. Ce
  pré-calcul fait sur place est un reduce partiel, que Dean et Ghemawat (2004)
  appellent un *combiner* ;
- **reduce** : on additionne les agrégats partiels, puis on divise à la toute fin.

**Une difficulté propre à DVF.** La règle 3 du nettoyage a besoin de **toutes les
lignes d'une vente** à la fois. Or `read_csv(..., chunksize=200_000)` coupe le
fichier toutes les 200 000 lignes, n'importe où : une vente peut se retrouver à
cheval sur deux morceaux. La fonction `par_morceaux` (fournie) évite cela : elle
reporte la dernière vente de chaque morceau au morceau suivant. Elle suppose que
les lignes d'une même vente se suivent dans le fichier, ce qui est le cas ici.

In [59]:
def par_morceaux(fichier, taille=200_000):
    """Lit le fichier par morceaux, sans jamais couper une vente en deux."""
    reste = None
    for morceau in pd.read_csv(fichier, usecols=COLS, dtype=TYPES, chunksize=taille):
        if reste is not None:
            morceau = pd.concat([reste, morceau])
        coupee = morceau["id_mutation"] == morceau["id_mutation"].iloc[-1]
        reste = morceau[coupee]            # la dernière vente continue peut-être plus loin
        yield morceau[~coupee]
    yield reste

In [74]:
t0 = time.time()
partiels = []

for morceau in par_morceaux(fichier_fr):
    propre = nettoie(morceau)
    par_dep = (propre.groupby("code_departement")["prix_m2"]
                        .agg(n="size", total="sum")
                        .query("n >= 50").sort_values(by="code_departement", ascending=False))

    partiels.append(par_dep)
    # MAP : nettoyer le morceau, puis calculer n et la somme des prix_m2
    #       par code_departement ; ajouter le résultat à `partiels`


# REDUCE : fusionner les agrégats partiels, puis calculer la moyenne
combine = pd.concat(partiels).groupby(level=0).sum()
print(combine.head())
combine["prix_m2_moyen"] = combine.total / combine.n

print(f"{len(partiels)} morceaux traités en {time.time() - t0:.1f} s")
combine.sort_values("prix_m2_moyen", ascending=False).head(8).round(0)

                     n         total
code_departement                    
01                7554  2.198225e+07
02                5667  7.652830e+06
03                4843  6.427454e+06
04                3035  7.559875e+06
05                3037  9.046698e+06
21 morceaux traités en 10.5 s


,n,total,prix_m2_moyen
code_departement,,,
75,27271,279895691.0,10263.0
92,16594,119444210.0,7198.0
94,12873,70182024.0,5452.0
06,24294,125090061.0,5149.0
74,13023,63296496.0,4860.0
2A,1802,8150966.0,4523.0
83,21063,94219106.0,4473.0
93,12104,53962329.0,4458.0


**Question 3.3.** Pourquoi une vente coupée entre deux morceaux fausserait-elle
le résultat ? (Pensez à la règle 3.) Dans MapReduce, quelle étape garantit que
toutes les lignes d'une même clé se retrouvent sur la même machine ?

*Votre réponse :* car une vente coupée entre deux morceaux compterait comme deux ventes au lieu d'une. C'est la fonction par_morceaux qui permet ça

### 3.4 — Vérifier que le résultat est exact

Le traitement par morceaux n'est pas une approximation. Comparez au calcul direct
en mémoire sur le DataFrame `france`.

In [75]:
direct = nettoie(france).groupby("code_departement")["prix_m2"].mean()
ecart = (direct - combine["prix_m2_moyen"]).abs().max()
print("écart maximum entre les deux méthodes :", ecart)
print("=> identique à la précision machine près" if ecart < 1e-6 else "=> ATTENTION, écart réel")

écart maximum entre les deux méthodes : 9.094947017729282e-13
=> identique à la précision machine près


### 3.5 — Et la médiane ?

Reprenez exactement la même stratégie, mais pour la **médiane** : calculez la
médiane par département sur chaque morceau, puis faites la médiane des médianes.
Comparez au vrai résultat.

In [ ]:
medianes_partielles = []
for morceau in par_morceaux(fichier_fr):
    # à compléter : médiane des prix_m2 par département, sur ce morceau
    ...

mediane_des_medianes = ...      # médiane des médianes partielles
vraie_mediane = ...             # médiane calculée sur `france` en une fois

comparaison = pd.DataFrame({"mediane_des_medianes": mediane_des_medianes,
                            "vraie_mediane": vraie_mediane})
comparaison["ecart"] = (comparaison.mediane_des_medianes - comparaison.vraie_mediane).abs()
print("écart maximum :", round(comparaison.ecart.max(), 1), "euros/m²")
print("départements sans aucun écart :", (comparaison.ecart == 0).sum(), "sur", len(comparaison))

**Question 3.5 — la plus importante du TP.**

- Pour la **moyenne**, les deux méthodes donnent le même résultat. Pour la
  **médiane**, non. Expliquez pourquoi, avec vos mots.
- Quelle propriété doit avoir une statistique pour être calculable par morceaux ?
- L'**écart-type** se calcule-t-il par morceaux ? Si oui, quels agrégats partiels
  faut-il garder pour chaque département ? (Indice : la variance est la moyenne des
  carrés moins le carré de la moyenne.)
- Pour la plupart des départements, la médiane des médianes tombe pourtant **juste**
  (dernière ligne affichée). Pourquoi ceux-là, et pas les autres ? (Indice : dans
  quel ordre les lignes sont-elles rangées dans le fichier ?)
- Que faudrait-il faire, concrètement, pour obtenir la vraie médiane sans jamais
  tout charger en mémoire ?

*Votre réponse :*
Car la médiane n'est pas décomposable : elle va dépendre de la découpe choisie
La propriété doit être indépendante du découpage, donc doit être linéaire
L'ecart type n'est pas linéaire, donc on ne peut pas le calculer par morceaux
Si on trie la liste des ventes en prix_m2 croissant ou décroissant, on pourrait avoir une

> C'est exactement la difficulté que rencontrent Hadoop et Spark. Spark sait
> calculer une médiane exacte (`median`, depuis sa version 3.4), mais elle coûte
> cher sur de très grands volumes : d'où `percentile_approx`, une médiane
> **approchée**.

---
## Pour aller plus loin (facultatif)

**Vers QGIS.** DVF contient `longitude` et `latitude`. Exportez un CSV
de points avec le prix au m², et vous pourrez le charger comme couche dans QGIS
pour produire une carte — une excellente base de figure pour le dossier.
Ce fichier contient des données personnelles : il reste sur votre machine,
ne le publiez pas.

```python
carte = nettoie(df)[["longitude", "latitude", "prix_m2", "nom_commune", "type_local"]]
carte.dropna(subset=["longitude", "latitude"]).to_csv("dvf69_points.csv", index=False)
```

**Citer le jeu de données.** Pour une figure du dossier de synthèse, la source se cite
comme au cours 2 : *DGFiP ; data.gouv.fr. Demandes de valeurs foncières géolocalisées,
année 2023 (millésime 2025-12). https://files.data.gouv.fr/geo-dvf/2025-12/csv/2023/.
Mis à jour le 25 mai 2026. Téléchargé le [date]. Licence Ouverte 2.0.* Rappel : dans le
dossier, la figure doit être produite avec **SPARQL ou QGIS**, pandas n'intervenant
qu'en appui.

**Vers Spark.** L'API de Spark s'écrit `df.groupBy("code_departement").agg(...)`.
Le vocabulaire que vous venez d'utiliser est déjà le bon : il ne reste qu'à
changer de moteur d'exécution. C'est l'objet du module **MSO 3.1**.

---

### Avant le QCM (gardez 15 min)

Il n'y a **rien à rendre**. Gardez votre notebook : il vous servira à réviser pour
l'examen final. Sur Colab : *Fichier → Enregistrer une copie dans Drive*.

Puis **fermez l'ordinateur**. Le QCM compte **15 questions**, dont 5 plus difficiles (★), et vérifie que vous savez :

- expliquer pourquoi **une ligne n'est pas une vente**, et ce que cela change au prix au m² ;
- **justifier** une règle de nettoyage ;
- **estimer** la mémoire d'un tableau avant de le charger, et dire pourquoi il pèse plus que son fichier ;
- **lire** un `groupby` et le relier à *map*, *shuffle* et *reduce* ;
- dire quelles statistiques se calculent **par morceaux**, et lesquelles non ;
- traiter un identifiant comme du **texte**, et **lire** un message d'erreur ;
- *(questions ★)* calculer une **variance** exacte à partir de résumés par morceau, prévoir ce que devient une **vente coupée** entre deux morceaux,
  dire **combien de départements** une médiane par morceaux peut fausser, estimer la **mémoire d'un morceau** lu avec `usecols`,
  et repérer un **bogue** dans un calcul par morceaux (map et reduce).